# 동화 캐릭터·배경·동작 SDXL LoRA 학습

이 노트북은 `assets/characters`의 캐릭터 PNG를 장르별 배경과 합성하고, 캐릭터 토큰과 동작 캡션을 포함한 데이터셋을 만든 뒤 SDXL LoRA를 학습합니다.

- Colab에서 **런타임 > 런타임 유형 변경 > T4 GPU**를 먼저 선택하세요.
- Hugging Face 토큰은 Colab의 **보안 비밀(Secrets)**에 `HF_TOKEN` 이름으로 저장하는 것을 권장합니다.
- 결과는 Google Drive의 `MyDrive/soft_capston_training`에 저장됩니다.
- 현재 10장은 시작용 데이터입니다. 캐릭터당 8~15개의 서로 다른 각도와 동작 원본을 추가하면 품질이 더 좋아집니다.
- 추가 캐릭터 이미지는 `source_characters/캐릭터키/`에 넣고, 같은 이름의 `.txt` 파일에 동작 설명을 적으면 자동 포함됩니다.
- 직접 준비한 배경은 `backgrounds/fantasy`, `backgrounds/adventure` 같은 장르 폴더에 넣으면 자동 생성 배경과 함께 사용됩니다.

> 이 LoRA는 SDXL용입니다. 현재 앱의 FLUX API에 그대로 적용되지는 않으며, 결과 확인 후 SDXL 추론 엔드포인트를 연결하거나 FLUX용 데이터셋으로 재사용할 수 있습니다.

In [ ]:
# 1. 학습 설정
REPO_URL = "https://github.com/Leejinhoe/soft_capston.git"
REPO_BRANCH = "main"
DIFFUSERS_TAG = "v0.38.0"

BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
BACKGROUND_MODEL = "stabilityai/sdxl-turbo"
DRIVE_FOLDER = "soft_capston_training"
USE_GOOGLE_DRIVE = True           # 실패하면 아래 설정에 따라 /content로 전환
ALLOW_LOCAL_STORAGE_FALLBACK = True

GENERATE_BACKGROUNDS = True       # False면 Drive의 backgrounds 폴더만 사용
BACKGROUNDS_PER_GENRE = 8         # 장르별 자동 생성 배경 수, 총 학습 이미지 80장
TRAIN_STEPS = 1200                # 빠른 확인 400, 권장 시작 1200
RESOLUTION = 512                  # T4 안전 설정
LORA_RANK = 16
LEARNING_RATE = 1e-4
RESUME_FROM_LATEST = True
REBUILD_DATASET = True
SEED = 42

# 학습할 캐릭터를 제한하려면 키를 남기세요. 빈 목록이면 5명 모두 학습합니다.
# 예: TRAIN_ONLY = ["fantasy_mina"]
TRAIN_ONLY = []


In [ ]:
# 2. GPU 확인 및 공식 Diffusers 학습 환경 설치
import os, shutil, subprocess, sys
from pathlib import Path

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu_name)
if "T4" not in gpu_name and "L4" not in gpu_name and "A100" not in gpu_name:
    print("주의: GPU 종류가 예상과 다릅니다. VRAM이 16GB 미만이면 OOM이 발생할 수 있습니다.")

# 저장소 폴더를 diffusers로 만들면 /content가 Python 경로에 있어 패키지 이름과 충돌합니다.
legacy_diffusers_dir = Path("/content/diffusers")
if (legacy_diffusers_dir / ".git").exists():
    shutil.rmtree(legacy_diffusers_dir)
DIFFUSERS_DIR = Path("/content/hf_diffusers_repo")
if not DIFFUSERS_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", DIFFUSERS_TAG,
        "https://github.com/huggingface/diffusers.git", str(DIFFUSERS_DIR)
    ], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.38.0", "accelerate>=1.2.0",
    "transformers>=4.46.0,<5", "datasets>=3.1.0",
    "peft>=0.14.0", "bitsandbytes>=0.45.0", "xformers", "safetensors",
    "huggingface_hub>=0.27.0", "Pillow>=10.0.0"
], check=True)

# 이전 실패 셀에서 만들어진 잘못된 모듈 캐시를 제거합니다.
for module_name in list(sys.modules):
    if module_name == "diffusers" or module_name.startswith("diffusers."):
        del sys.modules[module_name]
import diffusers
from diffusers import AutoPipelineForText2Image
if diffusers.__version__ != "0.38.0":
    raise RuntimeError(f"Diffusers 버전이 잘못되었습니다: {diffusers.__version__}")
print("Diffusers:", diffusers.__version__, diffusers.__file__)

from accelerate.utils import write_basic_config
write_basic_config(mixed_precision="fp16")
print("환경 설치 완료")

In [ ]:
# 3. Google Drive, Hugging Face, 프로젝트 연결
import getpass
from google.colab import drive, userdata
from huggingface_hub import login

storage_base = Path("/content")
drive_connected = False
if USE_GOOGLE_DRIVE:
    try:
        drive.mount("/content/drive", force_remount=False, timeout_ms=120000)
        storage_base = Path("/content/drive/MyDrive")
        drive_connected = True
    except Exception as exc:
        print("Google Drive 연결 실패:", exc)
        if not ALLOW_LOCAL_STORAGE_FALLBACK:
            raise
        print("임시 저장소 /content를 사용합니다. 런타임 종료 전에 결과 ZIP을 내려받으세요.")

DRIVE_ROOT = storage_base / DRIVE_FOLDER
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if not hf_token:
    hf_token = getpass.getpass("Hugging Face Read 토큰 입력: ").strip()
if not hf_token:
    raise RuntimeError("HF_TOKEN이 필요합니다.")
login(token=hf_token, add_to_git_credential=False)

PROJECT_DIR = Path("/content/soft_capston")
if not PROJECT_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
        REPO_URL, str(PROJECT_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

required_asset_names = {
    "fantasy_mina.png", "fantasy_mina_magic.png",
    "adventure_jun.png", "adventure_jun_walking.png",
    "nature_sol.png", "nature_sol_helping.png",
    "friendship_hana.png", "friendship_hana_waving.png",
    "mystery_on.png", "mystery_on_investigating.png",
}
ASSET_DIR = PROJECT_DIR / "assets" / "characters"
project_assets_ready = ASSET_DIR.exists() and required_asset_names <= {
    path.name for path in ASSET_DIR.iterdir() if path.is_file()
}
if not project_assets_ready:
    fallback = DRIVE_ROOT / "source_characters"
    fallback.mkdir(parents=True, exist_ok=True)
    fallback_names = {path.name for path in fallback.iterdir() if path.is_file()}
    if not required_asset_names <= fallback_names:
        from google.colab import files
        print("GitHub/저장소에서 기본 캐릭터 10장을 찾지 못했습니다.")
        print("파일 선택 창에서 assets/characters의 PNG 10장을 모두 선택하세요.")
        uploaded = files.upload()
        for filename, content in uploaded.items():
            (fallback / Path(filename).name).write_bytes(content)
    ASSET_DIR = fallback
print("캐릭터 원본:", ASSET_DIR)
print("결과 폴더:", DRIVE_ROOT)
print("Google Drive 연결:", drive_connected)

In [ ]:
# 4. 캐릭터, 동작, 배경 학습 명세
CHARACTERS = {
    "fantasy_mina": {
        "token": "ftmina",
        "name": "Mina",
        "genre": "fantasy",
        "description": "Korean child with a short dark-brown bob, midnight-blue star cape, lavender tunic, brown boots, glowing star wand",
        "sources": {
            "fantasy_mina.png": "standing calmly, neutral expression",
            "fantasy_mina_magic.png": "casting a sparkling magic spell, joyful expression",
        },
    },
    "adventure_jun": {
        "token": "ftjun",
        "name": "Jun",
        "genre": "adventure",
        "description": "Korean child with tousled black hair, ochre field jacket, deep-red scarf, navy shorts, brown boots, canvas satchel, brass compass",
        "sources": {
            "adventure_jun.png": "standing with a compass, neutral expression",
            "adventure_jun_walking.png": "walking forward while checking a compass, determined expression",
        },
    },
    "nature_sol": {
        "token": "ftsol",
        "name": "Sol",
        "genre": "nature",
        "description": "Korean child with wavy dark-brown hair, moss-green capelet, leaf-pattern cream tunic, forest-brown boots, acorn pendant, wooden lantern",
        "sources": {
            "nature_sol.png": "standing with a wooden lantern, gentle expression",
            "nature_sol_helping.png": "kneeling to help a small forest friend, caring expression",
        },
    },
    "friendship_hana": {
        "token": "fthana",
        "name": "Hana",
        "genre": "friendship",
        "description": "Korean child with two low braids, sunflower-yellow cardigan, white blouse, denim-blue overalls, coral sneakers, friendship bracelet",
        "sources": {
            "friendship_hana.png": "standing with a warm smile",
            "friendship_hana_waving.png": "waving hello with a joyful expression",
        },
    },
    "mystery_on": {
        "token": "fton",
        "name": "On",
        "genre": "mystery",
        "description": "Korean child with neat short black hair, navy detective coat, amber scarf, charcoal trousers, brown shoes, magnifying glass, notebook",
        "sources": {
            "mystery_on.png": "standing with a magnifying glass, thoughtful expression",
            "mystery_on_investigating.png": "investigating a clue with a magnifying glass, curious expression",
        },
    },
}

BACKGROUND_PROMPTS = {
    "fantasy": [
        "enchanted forest clearing with glowing flowers and a distant crystal castle",
        "cozy wizard library with floating lanterns and star maps",
        "moonlit hill above a magical village with warm windows",
        "ancient stone gate covered in luminous vines",
    ],
    "adventure": [
        "sunny mountain trail with wooden signs and distant waterfalls",
        "wide grassland path leading toward ancient ruins",
        "coastal cliff trail with a small sailing ship below",
        "hidden jungle campsite beside a rope bridge",
    ],
    "nature": [
        "peaceful forest pond with fireflies and mossy stones",
        "spring meadow filled with wildflowers and small animal homes",
        "gentle woodland stream beneath large green leaves",
        "old tree sanctuary with warm lantern light",
    ],
    "friendship": [
        "bright neighborhood park prepared for a small picnic",
        "cozy classroom craft corner with colorful paper decorations",
        "sunset playground with warm lights and chalk drawings",
        "friendly village square during a small flower festival",
    ],
    "mystery": [
        "old library corridor with a trail of tiny glowing clues",
        "quiet museum gallery at dusk with an open display case",
        "rainy village street with warm lamps and mysterious footprints",
        "attic study filled with maps, clocks, and hidden drawers",
    ],
}

if TRAIN_ONLY:
    unknown = sorted(set(TRAIN_ONLY) - set(CHARACTERS))
    if unknown:
        raise ValueError(f"알 수 없는 캐릭터 키: {unknown}")
    CHARACTERS = {key: CHARACTERS[key] for key in TRAIN_ONLY}

EXTRA_SOURCE_ROOT = DRIVE_ROOT / "source_characters"
for character_key, config in CHARACTERS.items():
    extra_folder = EXTRA_SOURCE_ROOT / character_key
    config["extra_sources"] = []
    if not extra_folder.exists():
        continue
    for image_path in sorted(extra_folder.iterdir()):
        if image_path.suffix.lower() not in {".png", ".jpg", ".jpeg", ".webp"}:
            continue
        caption_path = image_path.with_suffix(".txt")
        action = (
            caption_path.read_text(encoding="utf-8").strip()
            if caption_path.is_file()
            else "performing a natural fairytale action with a clear expressive pose"
        )
        config["extra_sources"].append((image_path, action))

missing = [
    filename
    for config in CHARACTERS.values()
    for filename in config["sources"]
    if not (ASSET_DIR / filename).is_file()
]
if missing:
    raise FileNotFoundError(f"누락된 캐릭터 원본: {missing}")
source_count = sum(
    len(config["sources"]) + len(config["extra_sources"])
    for config in CHARACTERS.values()
)
print(f"학습 대상 {len(CHARACTERS)}명, 원본 {source_count}장")
print("추가 이미지 폴더:", EXTRA_SOURCE_ROOT)

In [ ]:
# 5. 장르별 배경 생성 또는 Drive 배경 재사용
import gc, random, torch
from diffusers import AutoPipelineForText2Image
from PIL import Image

BACKGROUND_ROOT = DRIVE_ROOT / "backgrounds"
BACKGROUND_ROOT.mkdir(parents=True, exist_ok=True)
needed_genres = sorted({config["genre"] for config in CHARACTERS.values()})

def existing_backgrounds(genre):
    folder = BACKGROUND_ROOT / genre
    folder.mkdir(parents=True, exist_ok=True)
    return sorted([
        path for path in folder.iterdir()
        if path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
    ])

needs_generation = GENERATE_BACKGROUNDS and any(
    len(existing_backgrounds(genre)) < BACKGROUNDS_PER_GENRE
    for genre in needed_genres
)

if needs_generation:
    background_pipe = AutoPipelineForText2Image.from_pretrained(
        BACKGROUND_MODEL,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )
    background_pipe.enable_model_cpu_offload()
    try:
        background_pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    for genre in needed_genres:
        folder = BACKGROUND_ROOT / genre
        current = existing_backgrounds(genre)
        prompts = BACKGROUND_PROMPTS[genre]
        for index in range(len(current), BACKGROUNDS_PER_GENRE):
            scene = prompts[index % len(prompts)]
            prompt = (
                "empty storybook background, no people, no characters, "
                "polished soft watercolor and gouache children's book illustration, "
                "clear foreground and depth, warm cinematic light, " + scene
            )
            generator = torch.Generator(device="cpu").manual_seed(SEED + index + len(genre) * 100)
            image = background_pipe(
                prompt=prompt,
                negative_prompt="person, child, character, text, watermark, logo, scary",
                num_inference_steps=4,
                guidance_scale=0.0,
                width=RESOLUTION,
                height=RESOLUTION,
                generator=generator,
            ).images[0]
            image.save(folder / f"generated_{index:02d}.jpg", quality=94)
            print("배경 저장:", genre, index + 1)

    del background_pipe
    gc.collect()
    torch.cuda.empty_cache()

for genre in needed_genres:
    count = len(existing_backgrounds(genre))
    if count == 0:
        raise RuntimeError(
            f"{genre} 배경이 없습니다. GENERATE_BACKGROUNDS=True로 바꾸거나 "
            f"{BACKGROUND_ROOT / genre}에 배경 이미지를 넣으세요."
        )
    print(f"{genre}: 배경 {count}장")

In [ ]:
# 6. 투명 캐릭터와 배경을 합성하고 캡션 데이터셋 생성
import json, math
from PIL import ImageFilter, ImageEnhance
from IPython.display import display

DATASET_DIR = DRIVE_ROOT / "datasets" / "fairytale_multi_v1"
if REBUILD_DATASET and DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

def crop_alpha(image):
    rgba = image.convert("RGBA")
    bbox = rgba.getchannel("A").getbbox()
    return rgba.crop(bbox) if bbox else rgba

def composite_character(character_path, background_path, output_path, variant):
    background = Image.open(background_path).convert("RGB").resize((RESOLUTION, RESOLUTION))
    character = crop_alpha(Image.open(character_path))
    randomizer = random.Random(SEED + variant * 97 + sum(map(ord, character_path.name)))
    target_height = int(RESOLUTION * randomizer.uniform(0.68, 0.82))
    target_width = max(1, int(character.width * target_height / character.height))
    character = character.resize((target_width, target_height), Image.Resampling.LANCZOS)

    max_x = max(0, RESOLUTION - target_width - 12)
    x = int(max_x * randomizer.uniform(0.18, 0.82)) if max_x else 0
    y = RESOLUTION - target_height - randomizer.randint(4, 18)

    shadow = Image.new("RGBA", background.size, (0, 0, 0, 0))
    alpha = character.getchannel("A")
    shadow_shape = Image.new("RGBA", character.size, (22, 18, 35, 105))
    shadow_shape.putalpha(alpha.point(lambda value: int(value * 0.38)))
    shadow.alpha_composite(shadow_shape, (x + 7, y + 9))
    shadow = shadow.filter(ImageFilter.GaussianBlur(8))

    canvas = background.convert("RGBA")
    canvas = Image.alpha_composite(canvas, shadow)
    canvas.alpha_composite(character, (x, y))
    canvas.convert("RGB").save(output_path, quality=95)

records = []
preview_paths = []
for character_key, config in CHARACTERS.items():
    backgrounds = existing_backgrounds(config["genre"])
    source_items = [
        (ASSET_DIR / filename, action)
        for filename, action in config["sources"].items()
    ] + config["extra_sources"]
    for source_index, (source_path, action) in enumerate(source_items):
        for background_index, background_path in enumerate(backgrounds):
            output_name = f"{character_key}_{source_index}_{background_index:02d}.jpg"
            output_path = DATASET_DIR / output_name
            composite_character(
                source_path,
                background_path,
                output_path,
                source_index * 100 + background_index,
            )
            scene = BACKGROUND_PROMPTS[config["genre"]][background_index % len(BACKGROUND_PROMPTS[config["genre"]])]
            caption = (
                f"{config['token']} character, {config['name']}, {config['description']}, "
                f"{action}, in {scene}, polished soft watercolor and gouache "
                "children's book illustration, rounded shapes, expressive face, no text"
            )
            records.append({"file_name": output_name, "text": caption})
            if len(preview_paths) < 12:
                preview_paths.append(output_path)

with (DATASET_DIR / "metadata.jsonl").open("w", encoding="utf-8") as file:
    for record in records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

minimum_recommended = len(CHARACTERS) * 16
print(f"학습 이미지: {len(records)}장")
if len(records) < minimum_recommended:
    print(f"권장량보다 적습니다. 최소 {minimum_recommended}장 이상을 권장합니다.")

thumbs = [Image.open(path).convert("RGB").resize((180, 180)) for path in preview_paths]
columns = 4
rows = math.ceil(len(thumbs) / columns)
grid = Image.new("RGB", (columns * 180, rows * 180), "white")
for index, thumb in enumerate(thumbs):
    grid.paste(thumb, ((index % columns) * 180, (index // columns) * 180))
display(grid)

In [ ]:
# 7. SDXL LoRA 학습
TRAIN_SCRIPT = DIFFUSERS_DIR / "examples" / "text_to_image" / "train_text_to_image_lora_sdxl.py"
OUTPUT_DIR = DRIVE_ROOT / "outputs" / "fairytale_sdxl_lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

command = [
    "accelerate", "launch", "--mixed_precision=fp16", str(TRAIN_SCRIPT),
    f"--pretrained_model_name_or_path={BASE_MODEL}",
    f"--train_data_dir={DATASET_DIR}",
    "--caption_column=text",
    f"--output_dir={OUTPUT_DIR}",
    f"--resolution={RESOLUTION}",
    "--center_crop",
    "--train_batch_size=1",
    "--gradient_accumulation_steps=4",
    "--gradient_checkpointing",
    "--use_8bit_adam",
    "--enable_xformers_memory_efficient_attention",
    f"--learning_rate={LEARNING_RATE}",
    "--lr_scheduler=cosine",
    "--lr_warmup_steps=100",
    f"--max_train_steps={TRAIN_STEPS}",
    "--checkpointing_steps=300",
    "--checkpoints_total_limit=2",
    f"--rank={LORA_RANK}",
    "--snr_gamma=5.0",
    "--dataloader_num_workers=2",
    "--report_to=none",
    f"--seed={SEED}",
    "--mixed_precision=fp16",
]
if RESUME_FROM_LATEST and any(OUTPUT_DIR.glob("checkpoint-*")):
    command.append("--resume_from_checkpoint=latest")

print("학습 시작. T4에서 설정에 따라 약 1~3시간 걸릴 수 있습니다.")
print(" ".join(map(str, command)))
subprocess.run(command, check=True)

weight_path = OUTPUT_DIR / "pytorch_lora_weights.safetensors"
if not weight_path.is_file():
    raise RuntimeError("LoRA 가중치가 생성되지 않았습니다. 위 학습 로그를 확인하세요.")
print("학습 완료:", weight_path, f"({weight_path.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
# 8. 학습 결과 샘플 생성
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

gc.collect()
torch.cuda.empty_cache()
sample_pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
sample_pipe.scheduler = DPMSolverMultistepScheduler.from_config(sample_pipe.scheduler.config)
sample_pipe.load_lora_weights(str(OUTPUT_DIR), weight_name="pytorch_lora_weights.safetensors")
sample_pipe.enable_model_cpu_offload()
try:
    sample_pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass

SAMPLE_DIR = OUTPUT_DIR / "samples"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
sample_images = []
for index, (character_key, config) in enumerate(CHARACTERS.items()):
    action = list(config["sources"].values())[-1]
    prompt = (
        f"{config['token']} character, {config['name']}, {config['description']}, "
        f"{action}, in a new {config['genre']} fairytale scene, polished soft watercolor "
        "and gouache children's book illustration, expressive face, full body, no text"
    )
    image = sample_pipe(
        prompt=prompt,
        negative_prompt="text, watermark, logo, blurry, duplicate person, scary, violent",
        num_inference_steps=24,
        guidance_scale=6.0,
        width=RESOLUTION,
        height=RESOLUTION,
        generator=torch.Generator(device="cpu").manual_seed(SEED + index),
    ).images[0]
    image.save(SAMPLE_DIR / f"{character_key}.png")
    sample_images.append(image.resize((256, 256)))

sample_grid = Image.new("RGB", (len(sample_images) * 256, 256), "white")
for index, image in enumerate(sample_images):
    sample_grid.paste(image, (index * 256, 0))
display(sample_grid)

del sample_pipe
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 9. 결과 설정 저장 및 ZIP 묶기
import zipfile
config_path = OUTPUT_DIR / "fairytale_training_config.json"
config_path.write_text(json.dumps({
    "base_model": BASE_MODEL,
    "diffusers_tag": DIFFUSERS_TAG,
    "resolution": RESOLUTION,
    "train_steps": TRAIN_STEPS,
    "lora_rank": LORA_RANK,
    "learning_rate": LEARNING_RATE,
    "characters": {
        key: {"token": value["token"], "name": value["name"], "genre": value["genre"]}
        for key, value in CHARACTERS.items()
    },
}, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = DRIVE_ROOT / "fairytale_sdxl_lora_result.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file() and "checkpoint-" not in str(path):
            archive.write(path, path.relative_to(OUTPUT_DIR))
print("최종 결과:", zip_path)
print("트리거 토큰:", {key: value["token"] for key, value in CHARACTERS.items()})
if not drive_connected:
    from google.colab import files
    print("Drive가 연결되지 않아 결과 ZIP 다운로드를 시작합니다.")
    files.download(str(zip_path))

## 결과 판단 기준

1. 얼굴·머리·옷·소품이 원본 캐릭터와 비슷한지 확인합니다.
2. 동작을 바꿔도 캐릭터가 유지되는지 확인합니다.
3. 배경이 바뀌었는데 캐릭터가 배경 색에 과도하게 물들지 않는지 확인합니다.
4. 원본 포즈만 반복되면 원본 동작 이미지를 더 추가해야 합니다.
5. 캐릭터가 섞이면 캐릭터별로 `TRAIN_ONLY`를 지정해 별도 LoRA를 만드는 편이 좋습니다.

학습 결과가 괜찮으면 다음 단계에서 FastAPI가 SDXL+LoRA 추론 엔드포인트를 호출하도록 연결합니다. 현재 Hugging Face FLUX 라우터에 이 SDXL LoRA 파일을 직접 전달할 수는 없습니다.